# MIRAGE FlowPic Classification
di Mario Gabriele Carofano

### Panoramica
### Dataset
### Output

---

## Setup ambiente e riproducibilità

In [ ]:
#	LIBRARIES
#   ####################################################################    #

# Importing constant values
import importlib
import sys
sys.path.insert(1, '../src/')
import constants
importlib.reload(constants)

# Importing traffic converter functions
from traffic_converter import *

In [ ]:
#	FUNCTIONS
#   ####################################################################    #

def get_dataset_name(dir_path, dataset_name, debug=False):
    """ Estrae i metadati dal nome del file PICKLE e costruisce il nome del file .npy
    in cui salvare il dataset di istogrammi 2D (FlowPics).

    Args:
        dir_path (str): Percorso della cartella in cui salvare il file .npy.
        dataset_name (str): Nome del file PICKLE elaborato per costruire il dataset. Viene utilizzato per estrarre metadati e costruire il nome del file .npy.
        debug (bool, optional): Se True, stampa informazioni di debug durante il salvataggio. Defaults to False.
    """

    #   ####################################################################    #
    #   INIZIALIZZAZIONE

    if debug:
        print(f"[DEBUG] Cartella di salvataggio: {dir_path}")
        print(f"[DEBUG] Nome del file originale: {dataset_name}")

    counter = 0
    campi = []
    indices = [
        0, # dataset name
        4, # number of packets
        5, # number of features
        6, # traffic type
        8, # padding
        # 9  # hash code
    ]

    #   ####################################################################    #
    #   ESTRAZIONE DEI METADATI DAL NOME DEL FILE

    ret_name = dataset_name.split('.')[0]
    metadata = ret_name.split('_')

    for idx in indices:
        if idx < len(metadata) and metadata[idx]:
            campi.append(metadata[idx])
            counter += 1
        elif debug:
            print(f"[DEBUG] Campo metadata[{idx}] non trovato.")
    
    if counter == len(indices):
        ret_name = "_".join(campi)

    #   ####################################################################    #
    #   RITORNO DEL NOME DEL FILE

    if debug:
        print(f"[DEBUG] Nome del percorso completo di salvataggio: {dir_path}{ret_name}.npy")
    
    return f"{dir_path}{ret_name}"

    # end

def print_nonzero_elements(flow : np.ndarray) -> None:
	""" Stampa gli elementi non nulli di un array NumPy (FlowPic),
	mostrando le coordinate (riga, colonna) e il valore corrispondente.

	Args:
		flow (numpy.ndarray): Array 2D (FlowPic) da cui estrarre gli elementi non nulli.
	"""
    
	rows, cols = np.nonzero(flow)
	values = flow[rows, cols]

	print("Coordinate degli elementi non nulli:")
	for r, c, v in zip(rows, cols, values):
		print(f"[{r},{c}] = {v}", end=" | ")

	print()

	# end



---

In [ ]:
importlib.reload(constants)
from constants import (
    DATA_PATH, DATASET_NAME,
    MIN_TPS, MIN_PACKETS, MIN_DIM
)

#   ####################################################################    #

filters = {
	'min_tps': MIN_TPS,
	# 'min_dim': MIN_DIM,
	# 'min_packets': MIN_PACKETS,
}

debug = False
debug_cycle = False
flows_to_inspect = None

dataset, metadata = mirage_pickle_converter(
	DATA_PATH + DATASET_NAME,
	filters, debug, debug_cycle, flows_to_inspect
)

if debug and flows_to_inspect is not None:
	
	for fid in flows_to_inspect:

		flow_metadata = metadata.loc[metadata['FlowID'] == fid]

		if flow_metadata.empty:
			print(f"[DEBUG] Flusso n.{fid} non trovato.")
			continue

		did = flow_metadata['DatasetID'].values[0]

		print(f"[DEBUG] Elaborazione del flusso n.{fid}")
		print(f"Shape: {dataset[did].shape}")
		print(f"Metadata:\n{flow_metadata.to_string(index=False)}")
		print_nonzero_elements(dataset[did][0])

		print(f"Min: {np.min(dataset[did])}")
		print(f"Max: {np.max(dataset[did])}")
		print(f"Mean: {np.mean(dataset[did])}")
		print(f"Std: {np.std(dataset[did])}")
		print(f"Sum: {np.sum(dataset[did])}\n")

		# end for fid
	# end if

np.save(
    get_dataset_name(DATA_PATH, DATASET_NAME, debug),
    dataset
)